# Creating a Simple Agent with Tracing

In [4]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [7]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [8]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

In [9]:
sports_agent = Agent(
    name="Sports Assistant",
    instructions="""
    You are a helpful assistant giving out sports advice.
    You give concise answers.
    """,
)

In [5]:
with trace("Simple sports Agent"):
    result = await Runner.run(sports_agent, "What are the benefits of running?")

print(result)

NameError: name 'trace' is not defined

Let's execute the Agent:

In [7]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy and convenient. Quick facts:
    
    - Nutrients: good source of potassium, vitamin B6, vitamin C, and dietary fiber.
    - Calories: about 100–110 per medium banana.
    - Carbs/sugar: mainly naturally occurring sugars; still count toward daily carb intake.
    - Benefits: supports heart/kidney health, energy for workouts, can help with satiety and gut health (fiber).
    - Tip: choose ripe for easier digestion; unripe has more resistant starch.
    
    Cautions:
    - if you have kidney disease or must limit potassium, or are watching sugar/carb intake, portion size matters.
    
    Overall: a nutritious, everyday fruit when eaten in moderation.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [11]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are apple and guava?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Both are healthy parts of a balanced diet.

- Apple (with skin): good fiber (esp. pectin), moderate vitamin C, polyphenols. About 95 kcal per medium apple.
- Guava (raw): very high vitamin C and fiber, plus potassium and antioxidants. Much lower calories per 100 g than many fruits.

Tips:
- Eat whole with skin for max fiber.
- Guava is an excellent vitamin C boost; apples are great for fullness and polyphenols.
- Watch portion if managing sugar intake.

_Good Job!_